# SpaceX Falcon 9 First Stage Landing Prediction

## Refreshing the dataset with launches since 2020

Notebooks 01-07 use data through 2020-11-13 -- 90 Falcon 9 launches. SpaceX has flown hundreds more
since then. Here I bring the dataset up to date and see whether more data changes any of the
conclusions from notebook 07, especially the overfitting risk flagged there (permutation importance
dominated by near-unique `Serial`/`LandingPad` columns).

**Why Wikipedia instead of the SpaceX API**: the public SpaceX API (`api.spacexdata.com`) has been down
throughout this project (repeated `525` errors, on every version path including the nonexistent `v5` --
the latest real release is `v4`). Wikipedia's launch tables are live, actively maintained, and I already
built a scraper for them in notebook 02. Reusing that approach instead of waiting on an external service
that's shown no sign of recovering keeps this notebook actually runnable.

## What this notebook does

- Scrape the live Wikipedia launch tables for 2020 onward
- Clean and label the new launches the same way as notebooks 01-05
- Combine them with the original 90 launches into an extended dataset
- Save the result as new files, without touching the original `dataset_part_1.csv` / `dataset_part_3.csv`
  the rest of the pipeline depends on

## Scraping launches since 2020

Since the 2021 snapshot notebook 02 used was taken, Wikipedia has split this list into year-range
sub-pages. 2025-2026 launches are still on the main page; earlier years live on their own pages. I scrape
all of them and combine the results.

In [1]:
from spacex_capstone.scraping import scrape_multiple_pages

wikipedia_urls = [
    "https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches_(2020%E2%80%932022)",
    "https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches_(2023)",
    "https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches_(2024)",
    "https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches",  # 2025-2026
]

raw_scraped = scrape_multiple_pages(wikipedia_urls)
print(f"Scraped {len(raw_scraped)} launch rows across {len(wikipedia_urls)} pages.")
raw_scraped.head()

Scraped 604 launch rows across 4 pages.


,FlightNumber,Date,Time,BoosterVersion,LaunchSite,Payload,PayloadMass,Orbit,Customer,Outcome,BoosterLanding
0,78,7 January 2020,02:19:21,F9 B,Cape Canaveral,Starlink,15600.0,LEO,SpaceX,Success,Success (
1,79,19 January 2020,15:30,F9 B,Kennedy,Crew Dragon in-flight abort test,12050.0,Sub-orbital,NASA,Successful,No attempt
2,80,29 January 2020,14:07,F9 B,Cape Canaveral,Starlink,15600.0,LEO,SpaceX,Success,Success (
3,81,17 February 2020,15:05,F9 B,Cape Canaveral,Starlink,15600.0,LEO,SpaceX,Success,Failure (
4,82,7 March 2020,04:50,F9 B,Cape Canaveral,SpaceX CRS-20,1977.0,LEO,NASA,Success,Success (


## Cleaning and filtering

I parse the `Date` column, restrict to launches strictly after the original dataset's cutoff
(2020-11-13) to avoid double-counting, and confirm every scraped row is actually a Falcon 9 (not
Falcon Heavy) launch.

**Note on date parsing**: `pd.to_datetime` needs `format="mixed"` here. Wikipedia's older pages write
dates as "7 January 2020" while the newer main-page rows use "August 16, 2026" -- mixing both formats in
one column makes pandas' format auto-detection guess wrong for a large fraction of rows (it infers a
single format from the first value and applies it column-wide) unless told explicitly to handle each
row on its own terms.

In [2]:
import pandas as pd

raw_scraped["ParsedDate"] = pd.to_datetime(raw_scraped["Date"], errors="coerce", format="mixed")
print(f"Unparseable dates: {raw_scraped['ParsedDate'].isna().sum()} (should be 0)")

ORIGINAL_CUTOFF_DATE = "2020-11-13"
new_launches = raw_scraped[raw_scraped["ParsedDate"] > ORIGINAL_CUTOFF_DATE].copy()
print(f"New launches after the original cutoff: {len(new_launches)}")

# All rows should be Falcon 9 ("F9 ..."); this is a sanity check, not a filter --
# this specific launch list is Falcon 9-only to begin with, unlike the combined
# API data in notebooks 01/03 which needed an explicit Falcon 1 filter.
assert new_launches["BoosterVersion"].str.startswith("F9").all(), "Unexpected non-Falcon-9 booster version found"
new_launches[["FlightNumber", "ParsedDate", "LaunchSite", "PayloadMass", "Orbit", "BoosterLanding"]].head()

Unparseable dates: 0 (should be 0)
New launches after the original cutoff: 584


,FlightNumber,ParsedDate,LaunchSite,PayloadMass,Orbit,BoosterLanding
20,98,2020-11-16,Kennedy,0.0,LEO,Success (
21,99,2020-11-21,Vandenberg,1192.0,LEO,Success (
22,100,2020-11-25,Cape Canaveral,15600.0,LEO,Success (
23,101,2020-12-06,Kennedy,2972.0,LEO,Success (
24,102,2020-12-13,Cape Canaveral,7000.0,GTO,Success (


## Building the landing-success label

I use `build_landing_class_from_text` (`src/spacex_capstone/features.py`) rather than
`build_landing_class`: Wikipedia's `Booster landing` cell text gets truncated by the HTML parser (its
`<th>` -- and therefore the pattern the row-parsing logic relies on -- differs from the neat enumerated
strings the SpaceX API returns), so I match on whether the text starts with "Success" instead of an
exact lookup table.

In [3]:
from spacex_capstone.features import build_landing_class_from_text

new_launches["Class"] = build_landing_class_from_text(new_launches["BoosterLanding"])
print(new_launches["Class"].value_counts())
print(f"Success rate in the new data: {new_launches['Class'].mean():.1%}")

Class
1    573
0     11
Name: count, dtype: int64
Success rate in the new data: 98.1%


## A real data-quality issue: missing payload mass

**81% of the newly scraped rows have no payload mass listed** (Wikipedia mostly stopped recording
individual masses once Starlink batch launches became the majority of Falcon 9 flights). Treating those
as literal zeros would badly distort the feature -- a `PayloadMass` column that's mostly 0 isn't
"light payloads," it's missing data. I treat 0 the same way notebook 01 treats a `NaN`: as missing,
imputed with the mean of the values that *are* known -- consistent with the reasoning already used for
the original dataset, not a new, unrelated choice.

In [4]:
print(f"Rows with PayloadMass == 0: {(new_launches['PayloadMass'] == 0).sum()} / {len(new_launches)} "
      f"({(new_launches['PayloadMass'] == 0).mean():.0%})")

payload_mass_mean = new_launches.loc[new_launches["PayloadMass"] > 0, "PayloadMass"].mean()
new_launches["PayloadMass"] = new_launches["PayloadMass"].replace(0, payload_mass_mean)
print(f"Imputed missing PayloadMass with the mean of known values: {payload_mass_mean:.0f} kg")

Rows with PayloadMass == 0: 487 / 584 (83%)
Imputed missing PayloadMass with the mean of known values: 6182 kg


## Coarsening launch site names to combine with the original data

Wikipedia's launch-site links now point to the base facility ("Cape Canaveral", "Kennedy",
"Vandenberg") rather than the specific pad code the SpaceX-API-derived dataset used ("CCAFS SLC-40",
"KSC LC-39A", "VAFB SLC-4E"). To combine both sources into one feature space, I coarsen the original
data's site names down to the same facility level with `normalize_launch_site` -- this loses the
pad-level distinction for the original 90 launches, a real trade-off worth being explicit about rather
than silently reducing everyone's granularity without saying so.

In [5]:
from spacex_capstone.data import load_reference_dataset
from spacex_capstone.features import normalize_launch_site

original = load_reference_dataset("../data/raw/dataset_part_1_reference.csv")
original_comparable = pd.DataFrame({
    "FlightNumber": original["FlightNumber"],
    "PayloadMass": original["PayloadMass"],
    "Orbit": original["Orbit"],
    "LaunchSite": normalize_launch_site(original["LaunchSite"]),
    "Class": original["Class"] if "Class" in original.columns else None,
})

# The reference CSV predates the Class column in some snapshots -- rebuild it
# the same way notebook 03 does if it's missing, so this cell works either way.
if original_comparable["Class"].isna().all():
    from spacex_capstone.features import build_landing_class
    bad_outcomes = {"None None", "False ASDS", "False Ocean", "None ASDS", "False RTLS"}
    original_comparable["Class"] = build_landing_class(original["Outcome"], bad_outcomes=frozenset(bad_outcomes))

original_comparable.head()

,FlightNumber,PayloadMass,Orbit,LaunchSite,Class
0,1,6104.959412,LEO,Cape Canaveral,0
1,2,525.000000,LEO,Cape Canaveral,0
2,3,677.000000,ISS,Cape Canaveral,0
3,4,500.000000,PO,Vandenberg,0
4,5,3170.000000,GTO,Cape Canaveral,0


## Combining into one extended dataset

In [6]:
new_comparable = pd.DataFrame({
    "FlightNumber": new_launches["FlightNumber"],
    "PayloadMass": new_launches["PayloadMass"],
    "Orbit": new_launches["Orbit"],
    "LaunchSite": new_launches["LaunchSite"],
    "Class": new_launches["Class"],
})

extended = pd.concat([original_comparable, new_comparable], ignore_index=True)
extended = extended.dropna(subset=["Orbit", "LaunchSite"])
print(f"Extended dataset: {len(extended)} launches ({len(original_comparable)} original + "
      f"{len(new_comparable)} new, {len(new_comparable) - (len(extended) - len(original_comparable))} "
      f"dropped for missing Orbit/LaunchSite)")
print(f"Overall success rate: {extended['Class'].mean():.1%}")

extended.to_csv("../data/dataset_part_1_extended.csv", index=False)
extended.head()

Extended dataset: 672 launches (90 original + 584 new, 2 dropped for missing Orbit/LaunchSite)
Overall success rate: 93.9%


,FlightNumber,PayloadMass,Orbit,LaunchSite,Class
0,1,6104.959412,LEO,Cape Canaveral,0
1,2,525.000000,LEO,Cape Canaveral,0
2,3,677.000000,ISS,Cape Canaveral,0
3,4,500.000000,PO,Vandenberg,0
4,5,3170.000000,GTO,Cape Canaveral,0


## Feature engineering: one-hot encoding

Only `FlightNumber`, `PayloadMass`, `Orbit`, and `LaunchSite` are available in both sources, so that's
the feature set here -- a real reduction from the original 83 one-hot columns (no `Serial`,
`LandingPad`, `Block`, `GridFins`, `Reused`, `Legs`, `ReusedCount`: those come only from the SpaceX
API's per-core enrichment data, which Wikipedia doesn't have). That's not just an unavoidable
limitation -- it's arguably a *more honest* feature set: a booster's `Serial` number is a near-unique
identifier notebook 07 already flagged as a likely overfitting signal, and it's not something anyone
could supply for a *future*, not-yet-flown launch anyway (which is exactly what the prediction endpoint
in a later step needs to work with).

In [7]:
from spacex_capstone.features import one_hot_encode_features

features_extended = one_hot_encode_features(
    extended[["FlightNumber", "PayloadMass", "Orbit", "LaunchSite"]],
    categorical_columns=["Orbit", "LaunchSite"],
)
features_extended["Class"] = extended["Class"].astype("float64")

print(f"Extended feature table: {features_extended.shape[0]} rows x {features_extended.shape[1]} columns")
features_extended.to_csv("../data/dataset_part_3_extended.csv", index=False)
features_extended.head()

Extended feature table: 672 rows x 24 columns


,FlightNumber,PayloadMass,Orbit_BLT,Orbit_Ballistic lunar transfer (BLT),Orbit_ES-L1,Orbit_GEO,Orbit_GTO,Orbit_HEO,Orbit_Heliocentric,Orbit_ISS,...,Orbit_Polar,Orbit_Retrograde,Orbit_SO,Orbit_SSO,Orbit_TLI,Orbit_VLEO,LaunchSite_Cape Canaveral,LaunchSite_Kennedy,LaunchSite_Vandenberg,Class
0,1.0,6104.959412,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,2.0,525.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,3.0,677.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
3,4.0,500.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,5.0,3170.000000,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


## Summary

This gives an extended dataset with far more launches than the original 90, at the cost of a smaller,
coarser feature set (facility-level launch site, no per-booster identifiers) and a payload-mass column
that's mostly imputed rather than measured. Both trade-offs are used and discussed further in
`07_machine_learning_prediction.ipynb`'s retraining section, where I compare model performance and
feature importance on this dataset against the original.